[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/okumura-kobe/kobe-data-business/blob/main/lecture08/lecture8_notebook.ipynb)

# 後期大学院演習「データで動かす社会とビジネス」
## 第8回　演習 Notebook

**テーマ：デフォルト確率を損失・収益・判断に変える**
― モデルの違いは、誰の与信をどう変えるのか ―

---

### この Notebook の使い方

- **STEP 1〜5 の順番に実行**してください
- 各 STEP の結果をワークシートに記録してください
- 第6・7回で作った pd_A / pd_B をそのまま使います

> ⚠️ **今回行うこと**：PDをELに変換し、与信管理への影響を確認する
> ⚠️ **今回行わないこと**：Logit再推定・AUC再分析・Model B採用判断 → 第7回済み or 第9回

**今回の中心的な問い**
> Model B を使うと、与信管理はどう変わるのか？

---

| | |
|---|---|
| 学籍番号 | |
| 氏名 | |
| 実施日 | |

---
## Step 0　準備（変更不要）

In [ ]:
# ライブラリの読み込み（変更不要）
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['figure.figsize']    = (11, 4)

print('Ready')

In [ ]:
# データの読み込み（変更不要）
# ⚠️ 第8回では Logit を再推定しない
#    credit_model.csv に保存済みの pd_A / pd_B をそのまま使用する
#    （第6回でDevelopment 7,000件から推定した係数をValidationに適用した値）

url = "https://raw.githubusercontent.com/okumura-kobe/kobe-data-business/main/data/credit_model.csv"
df = pd.read_csv(url)

dev = df[df['split'] == 'development'].copy().reset_index(drop=True)
val = df[df['split'] == 'validation'].copy().reset_index(drop=True)

# 変数定義（参照用）
features_A = ['age', 'credit_limit', 'latest_payment_status',
              'latest_bill_amount', 'latest_payment_amount']
features_B = features_A + ['max_delay', 'months_with_delay',
              'avg_bill_amount', 'avg_payment_amount',
              'credit_utilization', 'payment_ratio']

print(f'Validation: {len(val):,}件')
print(f'pd_A 平均: {val["pd_A"].mean():.4f}  pd_B 平均: {val["pd_B"].mean():.4f}')
print(f'ead 平均: {val["ead"].mean():.0f} TWD  lgd: {val["lgd"].unique()}')

---
## STEP 1　PD を Expected Loss に変える（10分）

**EL = PD × LGD × EAD**

確率（PD）が残高（EAD）を通じて金額（EL）になります。
同じPD差でも、EADが違えば経済的影響は大きく変わります。

| 変数 | 設定 | 備考 |
|---|---|---|
| LGD | 0.45（固定） | 教材上の仮定値 |
| EAD | `ead`列 | `credit_limit × 0.7` |

In [ ]:
# 1-1　EL を計算する
LGD = 0.45  # 教材上の仮定値（実務のLGDとは異なる）

val['el_A'] = val['pd_A'] * LGD * val['ead']
val['el_B'] = val['pd_B'] * LGD * val['ead']
val['el_diff'] = val['el_B'] - val['el_A']

print('=== EL の基本統計（Validation）===')
print(f'{"指標":<20} {"el_A (Model A)":>16} {"el_B (Model B)":>16} {"el_diff (B-A)":>16}')
print(f'{"平均（TWD）":<20} {val["el_A"].mean():>16.1f} {val["el_B"].mean():>16.1f} {val["el_diff"].mean():>16.1f}')
print(f'{"合計（TWD）":<20} {val["el_A"].sum():>16.0f} {val["el_B"].sum():>16.0f} {val["el_diff"].sum():>16.0f}')
print(f'{"最小値":<20} {val["el_A"].min():>16.1f} {val["el_B"].min():>16.1f} {val["el_diff"].min():>16.1f}')
print(f'{"最大値":<20} {val["el_A"].max():>16.1f} {val["el_B"].max():>16.1f} {val["el_diff"].max():>16.1f}')

---
## STEP 2　顧客別の EL 変化を見る（10分）

平均・合計だけでは見えない顧客別の差を確認します。
ワークシートの **STEP 2** に上位10件を転記してください。

In [ ]:
# 2-1　EL 上昇幅の大きい顧客（上昇幅の大きい10件）
cols = ['customer_id', 'pd_A', 'pd_B', 'el_A', 'el_B', 'el_diff']

print('=== EL 上昇幅の大きい顧客（上昇幅の大きい10件）===')
print(val.nlargest(10, 'el_diff')[cols].to_string(index=False))

In [ ]:
# 2-2　EL 低下幅の大きい顧客（低下幅の大きい10件）
print('=== EL 低下幅の大きい顧客（低下幅の大きい10件）===')
print(val.nsmallest(10, 'el_diff')[cols].to_string(index=False))

---
## STEP 3　ポートフォリオ全体への影響を見る（10分）

個々の顧客ではなく、Validation 3,000件全体で集計します。
第5回で確認した履歴特徴量との関係を見ていきます。

In [ ]:
# 3-1　ポートフォリオ全体の EL 比較
print('=== ポートフォリオ全体の EL 比較（Validation）===')
print(f'EL 合計（Model A）: {val["el_A"].sum():>12,.0f} TWD')
print(f'EL 合計（Model B）: {val["el_B"].sum():>12,.0f} TWD')
print(f'変化（B − A）:      {val["el_diff"].sum():>12,.0f} TWD')
print()
print(f'EL 平均（Model A）: {val["el_A"].mean():>12,.1f} TWD')
print(f'EL 平均（Model B）: {val["el_B"].mean():>12,.1f} TWD')
print(f'変化（B − A）:      {val["el_diff"].mean():>12,.1f} TWD')

In [ ]:
# 3-2　履歴特徴量別の EL 差（セグメント集計）
# max_delay でグループ分けして el_diff の平均を確認
print('=== max_delay 別の el_diff 平均 ===')
seg = val.groupby('max_delay').agg(
    件数=('el_diff', 'count'),
    el_diff_平均=('el_diff', 'mean'),
    el_diff_合計=('el_diff', 'sum')
).round(1)
print(seg)

---
## STEP 4　与信枠への影響をシミュレーションする（10分）

教材上の仮想ルールを使って、Model A/B で与信区分がどう変わるかを確認します。

> ⚠️ T1 = 0.20・T2 = 0.40 は教材上の仮想しきい値。実務上の適正な Cut-off を示すものではない。

| 条件 | 与信区分 |
|---|---|
| `PD < 0.20` | 枠維持 |
| `0.20 ≤ PD < 0.40` | 枠縮小 |
| `PD ≥ 0.40` | 追加与信抑制 |

In [ ]:
# 4-1　与信区分を計算する
T1, T2 = 0.20, 0.40

def classify(pd_col):
    return pd.cut(
        val[pd_col],
        bins=[-np.inf, T1, T2, np.inf],
        labels=['枠維持', '枠縮小', '追加与信抑制'],
        right=False
    )

val['cat_A'] = classify('pd_A')
val['cat_B'] = classify('pd_B')

print('=== 与信区分別の件数（Validation）===')
print(f'{"与信区分":<12} {"Model A":>10} {"Model B":>10} {"変化(B-A)":>10}')
for cat in ['枠維持', '枠縮小', '追加与信抑制']:
    nA = (val['cat_A'] == cat).sum()
    nB = (val['cat_B'] == cat).sum()
    print(f'{cat:<12} {nA:>10,} {nB:>10,} {nB-nA:>+10,}')
print(f'{"合計":<12} {len(val):>10,} {len(val):>10,} {"0":>10}')

In [ ]:
# 4-2　Model A → B で区分が変わった顧客
changed = val[val['cat_A'] != val['cat_B']].copy()
print(f'=== 与信区分が変わった顧客: {len(changed):,}件（Validation {len(val):,}件中）===')
print()
print('変化の内訳:')
cross = val.groupby(['cat_A', 'cat_B'], observed=True).agg(
    件数=('customer_id', 'count'),
    実績Default率=('default', 'mean')
).round(4)
print(cross)

---
## STEP 5　統計的改善と与信管理上の影響を統合する（10分）

第7回（統計評価）と第8回（与信管理影響）の結果を合わせて整理します。

> ⚠️ ELが下がった＝良いモデル、ではない。
> Model B がより適切にリスクを捉えた結果、EL が上昇することもある。

ワークシートの **STEP 5** に結果を記録してください。

In [ ]:
# 5-1　統合サマリー
# AUCは第7回の確定値を使用（再計算しない）
AUC_A = 0.717
AUC_B = 0.759

print('=== 統合評価（Validation）===')
print(f'{"評価軸":<25} {"Model A":>12} {"Model B":>12} {"変化":>10}')
print(f'{"AUC（第7回確定値）":<25} {AUC_A:>12.3f} {AUC_B:>12.3f} {AUC_B-AUC_A:>+10.3f}')
print(f'{"EL 合計（TWD）":<25} {val["el_A"].sum():>12,.0f} {val["el_B"].sum():>12,.0f} {val["el_diff"].sum():>+10,.0f}')
print(f'{"与信区分変更件数":<25} {"―":>12} {(val["cat_A"]!=val["cat_B"]).sum():>12,} {"":>10}')

In [ ]:
# 5-2　統合解釈メモ（コメントとして記録してください）
#
# 【Model B によって何が改善し、どのような影響が生じるか】
#
# AUC・Calibrationの改善を与信管理の言葉で説明：
#
#
# ELの変化が意味すること：
#
#
# 与信区分変更の影響：
#
#
# 【第9回に持ち越す問い】
#
#


---
## まとめ

### 今日作ったもの

| STEP | 内容 |
|---|---|
| STEP 1 | EL = PD × LGD × EAD を計算した（el_A・el_B・el_diff） |
| STEP 2 | EL変化が大きい顧客を特定した（上昇・低下の上位10件） |
| STEP 3 | ポートフォリオ全体のEL差と履歴特徴量別の集計を確認した |
| STEP 4 | 仮想与信ルールで与信区分変更件数・変更顧客の特徴を確認した |
| STEP 5 | 統計的改善（AUC）と与信管理影響（EL・与信区分）を統合した |

### 今回の核心

> **確率は残高を通じて金額になる。**
> **数字が動くとき、動いているのは誰かの与信だ。**

---

### 第9回の予告：改訂判断と説明

- 第5回の判断基準（利用者保護・適正与信・EL・収益・改訂コスト）に戻る
- Model B を「維持 / 改訂 / 条件付き改訂」で判断する
- 金融庁監督局への説明資料を作る

**事前課題**
- 今日の統合評価表（STEP 5）をワークシートに記録しておく
- 第5回ワークシートの判断優先順位（8項目）を再確認しておく
- BEEF+ の第9回配布資料を読む

---
## Coding Challenge：Validation sample で EL と与信区分を自分で計算する

> **授業外の自習課題です。提出不要・採点対象外です。**（想定時間：30〜45分）
> 授業内の STEP 0〜5 の実行には影響しません。

授業中は完成コードを実行して、PD が EL と与信区分にどう変わるかを解釈しました。ここでは、STEP 1 の **EL の計算** と STEP 4 の **与信区分の作成** を、自分でコードを書いて再現します。

**前提**：STEP 0 を実行して、Validation sample の `val`（3,000件）を読み込み済みであること（`val` に `pd_A`・`pd_B`・`ead` がある）。

### 課題

`val` を使い、次の処理を自分で記述してください。

1. Model A・Model B の Expected Loss を、顧客ごとに計算する（`el_A_cc`・`el_B_cc`）
2. 両モデルの EL 合計を求め、Model B − Model A の差を計算する（`el_sum_A_cc`・`el_sum_B_cc`・`el_diff_total_cc`）
3. T1 = 0.20、T2 = 0.40 で、Model B の与信区分を作る（`cat_B_cc`）
   - PD < 0.20 は「枠維持」、0.20 ≤ PD < 0.40 は「枠縮小」、PD ≥ 0.40 は「追加与信抑制」
4. 与信区分別の件数を表示する
5. 結果を簡潔に解釈する（コメントで1〜2行）

> 授業内の `el_A`・`el_B`・`el_diff`・`cat_A`・`cat_B` を上書きしないよう、変数名は上の括弧内の名前を使ってください。`val` に新しい列は追加せず、別の変数に保存します。

### ヒント

- LGD は 0.45（固定）、EAD は `val` の `ead` 列を使います
- EL = PD × LGD × EAD です
- `pd.cut()` で区間に分けます。**境界の値（PD がちょうど 0.20 や 0.40）をどちらの区分に入れるか**は、`right` の指定で決まります
- 区分別の件数は、`value_counts()` や `groupby` で数えられます

授業内の STEP 1・STEP 4 のコードを見ながら、自分の言葉でコードを組み立ててみてください。

In [ ]:
# Coding Challenge：ここに自分でコードを書いてください
# （変数名：el_A_cc / el_B_cc / el_sum_A_cc / el_sum_B_cc / el_diff_total_cc / cat_B_cc）

# 1. Model A・Model B の Expected Loss を顧客ごとに計算する


# 2. 両モデルの EL 合計と、その差（Model B − Model A）を計算する


# 3. T1 = 0.20、T2 = 0.40 で Model B の与信区分を作る


# 4. 与信区分別の件数を表示する


# 5. 結果を簡潔に解釈する（コメントで1〜2行）
#


### 自己確認

上のセルで `el_A_cc`・`el_B_cc`・`el_sum_A_cc`・`el_sum_B_cc`・`el_diff_total_cc`・`cat_B_cc` を作ってから、次のセルを実行してください。次の5点を確認します。

- EL が顧客ごとに 3,000 件ずつ計算されている
- Model A の EL 合計が約 28,743,368 TWD である
- Model B の EL 合計が約 28,849,013 TWD である
- EL 合計の差（B − A）が約 +105,645 TWD である
- Model B の与信区分別件数が、枠維持 1,993／枠縮小 586／追加与信抑制 421 である

> EL の合計が増えても、全員の与信が悪くなったわけではありません。誰の EL が増え、誰の EL が減ったのかは、授業内の STEP 2・STEP 3 で確認しました。

In [ ]:
# Coding Challenge 自己確認（変更不要）
# el_A_cc / el_B_cc / el_sum_A_cc / el_sum_B_cc / el_diff_total_cc / cat_B_cc を作ってから実行してください
try:
    nA, nB = len(el_A_cc), len(el_B_cc)
    sA, sB, sD = float(el_sum_A_cc), float(el_sum_B_cc), float(el_diff_total_cc)
    cnt = pd.Series(cat_B_cc).value_counts()
    c1, c2, c3 = (int(cnt.get(k, 0)) for k in ['枠維持', '枠縮小', '追加与信抑制'])
except NameError:
    print('必要な変数がまだありません。上のセルで作成してから実行してください。')
else:
    def mark(ok): return 'OK' if ok else '要確認'
    print(f'EL の件数          : A {nA:,}件 / B {nB:,}件（期待値 各3,000件）  → {mark(nA == 3000 and nB == 3000)}')
    print(f'EL 合計（Model A） : {sA:,.0f} TWD（期待値 約28,743,368）  → {mark(abs(sA - 28743368) < 1)}')
    print(f'EL 合計（Model B） : {sB:,.0f} TWD（期待値 約28,849,013）  → {mark(abs(sB - 28849013) < 1)}')
    print(f'EL 合計の差（B−A） : {sD:+,.0f} TWD（期待値 約+105,645）  → {mark(abs(sD - 105645) < 1)}')
    print(f'Model B の与信区分 : 枠維持 {c1:,} / 枠縮小 {c2:,} / 追加与信抑制 {c3:,}'
          f'（期待値 1,993 / 586 / 421）  → {mark((c1, c2, c3) == (1993, 586, 421))}')
    # 参考：授業内の STEP 1・STEP 4 を実行済みなら、結果を比較する
    if 'el_B' in val.columns and 'cat_B' in val.columns:
        same = bool((cat_B_cc.astype(str).values == val['cat_B'].astype(str).values).all())
        print(f'（参考）授業内の cat_B と一致: {same}／EL 合計の差 {abs(sB - val["el_B"].sum()):.6f}')


---
### 解答例（自習後に確認してください）

**まず自分でコードを書いてから**、次のセルを開いてください。セルは折りたたまれています。Colab では、セルの「…」メニューの「コードを表示」で開けます。

解答例は、授業内の `el_A`・`el_B`・`cat_B` を使わず、別の変数名で EL と与信区分を最初から計算し直しています。

In [ ]:
#@title 解答例（自習後に確認する） { display-mode: "form" }
# 授業内の el_A / el_B / cat_B は使わず、別の変数名で EL と与信区分を最初から計算し直す

# 1. Model A・Model B の Expected Loss を顧客ごとに計算する
LGD_sol = 0.45  # 教材上の仮定値
el_A_sol = val['pd_A'] * LGD_sol * val['ead']
el_B_sol = val['pd_B'] * LGD_sol * val['ead']

# 2. 両モデルの EL 合計と、その差（Model B − Model A）を計算する
el_sum_A_sol = el_A_sol.sum()
el_sum_B_sol = el_B_sol.sum()
el_diff_total_sol = el_sum_B_sol - el_sum_A_sol

print(f'EL 合計（Model A）: {el_sum_A_sol:>12,.0f} TWD')
print(f'EL 合計（Model B）: {el_sum_B_sol:>12,.0f} TWD')
print(f'変化（B − A）:      {el_diff_total_sol:>+12,.0f} TWD')

# 3. T1 = 0.20、T2 = 0.40 で Model B の与信区分を作る
#    right=False：区間を「以上・未満」にする（PD = 0.20 は枠縮小、PD = 0.40 は追加与信抑制）
T1_sol, T2_sol = 0.20, 0.40
cat_B_sol = pd.cut(
    val['pd_B'],
    bins=[-np.inf, T1_sol, T2_sol, np.inf],
    labels=['枠維持', '枠縮小', '追加与信抑制'],
    right=False
)

# 4. 与信区分別の件数を表示する
print()
print('=== Model B の与信区分別件数（Validation）===')
print(cat_B_sol.value_counts().reindex(['枠維持', '枠縮小', '追加与信抑制']))

# 5. 結果についての解釈（例）
# ・Model B では EL 合計が Model A よりわずかに増える。
#   ただし合計の増減だけでなく、誰の EL がどう動くかを見る必要がある。
# ・与信区分は PD の水準で決まる。区分の境界の取り方（以上・未満）で、
#   境界上の顧客の扱いが変わるため、条件を明確に決めておく。
